# PromptGuard span detector: Colab GPU 학습

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/capstone-privai/promptguard-model/blob/main/notebooks/colab_train_span.ipynb)

이 노트북은 GPU에서 **학습만** 합니다. 경계 보정, cutoff 선택, 비교, gate 검사는 결과를 내려받아 로컬 CPU에서 기존 스크립트로 합니다. 판정 기준을 v9·v10과 같게 유지하기 위해서입니다. 자세한 절차는 `docs/ml_span/COLAB_TRAINING.md`에 있습니다.

1. 메뉴 **런타임 → 런타임 유형 변경 → GPU**를 고릅니다(T4 이상).
2. 아래 **설정** 셀만 고친 뒤 **런타임 → 모두 실행**을 누릅니다. Google Drive 연결을 묻는 창이 뜨면 허용합니다.
3. 학습 결과와 마지막 셀이 만든 zip은 Drive의 `promptguard-runs/`에 저장됩니다. zip을 내려받아 로컬에서 후처리합니다.

**지켜야 할 규칙**
- epoch과 cutoff는 validation에서만 고릅니다. 학습 스크립트는 끝에 test를 자동으로 평가해 `evaluation.json`과 `test-predictions.jsonl`을 만듭니다. cutoff를 고정하기 전에는 이 파일을 열지 마세요. 실수로 열지 않도록 zip에서도 뺐습니다.
- GPU가 없으면 노트북이 멈춥니다. CPU로 바꿔서 진행하지 않습니다.
- 가중치는 Git에 올리지 않습니다.

In [ ]:
# ── 설정: 여기만 고치세요 ──
RUN_NAME = 'span-mmbert-colab-v10warm-repro'  # 새 이름이어야 합니다. Drive에 같은 이름이 있으면 멈춥니다.
BASE_CONFIG = 'configs/span-mmbert-context-v10-warm.yaml'  # 재사용할 학습 레시피(데이터, 학습률, epoch 등)
START_FROM = 'v9'      # 'v9' 또는 'v10': models/의 bundle에서 이어 학습, 'backbone': mmBERT-small 원본에서 시작
OVERRIDES = {}         # BASE_CONFIG에서 바꿀 값. 예: {'batch_size': 16, 'learning_rate': 5e-6}
SEEDS = []             # 여러 seed로 차례로 학습하려면 예: [112, 113, 114]. 비우면 BASE_CONFIG의 seed로 한 번만 학습합니다.
REPO_REF = 'main'      # 학습에 쓸 코드와 데이터의 브랜치 또는 태그
PREPARE = []           # 학습 전에 저장소 폴더에서 실행할 셸 명령. 예: ['bash scripts/colab_prepare_v11.sh'] (git에 없는 데이터를 다시 만들 때)
DRIVE_DIR = '/content/drive/MyDrive/promptguard-runs'  # 결과 저장 위치(세션이 끊겨도 남습니다)

# 기본값은 v10 warm 학습(v9에서 이어 학습)을 GPU에서 그대로 재현합니다. 결과가 v10과 비슷한지 보며 Colab 환경을 먼저 확인하세요.

In [ ]:
# GPU 확인: 없으면 멈춥니다(CPU로 바꿔 진행하지 않음).
import torch
if not torch.cuda.is_available():
    raise RuntimeError('GPU가 없습니다. 런타임 → 런타임 유형 변경 → GPU를 고른 뒤 다시 실행하세요.')
GPU_NAME = torch.cuda.get_device_name(0)
print(GPU_NAME, round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1), 'GB')

In [ ]:
# Google Drive 연결: 결과를 Drive에 저장해 세션이 끊겨도 남깁니다.
import os
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Colab이 아니므로 Drive 연결을 건너뜁니다.')
os.makedirs(DRIVE_DIR, exist_ok=True)
# seed를 여러 개 주면 seed마다 RUN_NAME-seedN 폴더에 따로 저장합니다.
RUNS = [(f'{RUN_NAME}-seed{s}', s) for s in SEEDS] or [(RUN_NAME, None)]
for name, _ in RUNS:
    if os.path.exists(f'{DRIVE_DIR}/{name}') and os.listdir(f'{DRIVE_DIR}/{name}'):
        raise RuntimeError(f'{DRIVE_DIR}/{name} 가 이미 있습니다. RUN_NAME을 바꾸세요.')
# Drive가 차면 가중치 저장이 조용히 실패하고 학습이 끝난 뒤에야 드러납니다. run마다 가중치(0.6GB)와 zip(0.5GB)에 여유를 더해 2GB를 요구합니다.
import shutil
FREE_GB = shutil.disk_usage(DRIVE_DIR).free / 2**30
if FREE_GB < 2 * len(RUNS):
    raise RuntimeError(f'Drive 남은 공간이 {FREE_GB:.1f}GB입니다. run마다 2GB가 필요합니다. 옛 결과를 지우고 휴지통을 비우세요.')
print('결과 저장 위치:', *[f'{DRIVE_DIR}/{name}' for name, _ in RUNS], sep='\n  ')

In [ ]:
# 저장소 받기와 설치. 큰 가중치(LFS)는 필요한 파일 하나만 받습니다.
import subprocess

def sh(cmd):
    """Run a shell command and stream its output into the notebook."""
    print('$', cmd, flush=True)
    p = subprocess.Popen(['bash', '-o', 'pipefail', '-c', cmd], stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end='', flush=True)
    if p.wait():
        raise RuntimeError(f'명령 실패 (exit {p.returncode}): {cmd}')

REPO_URL = 'https://github.com/capstone-privai/promptguard-model.git'
REPO_DIR = '/content/promptguard-model'
if not os.path.exists(REPO_DIR):
    sh('git lfs version || (apt-get -qq update && apt-get -qq install -y git-lfs)')
    sh(f'GIT_LFS_SKIP_SMUDGE=1 git clone --depth 1 --branch {REPO_REF} {REPO_URL} {REPO_DIR}')
else:  # 같은 세션에서 다시 실행할 때 옛 코드로 학습하지 않도록 REPO_REF의 최신 커밋으로 바꿉니다.
    sh(f'cd {REPO_DIR} && GIT_LFS_SKIP_SMUDGE=1 git fetch -q --depth 1 origin {REPO_REF} && git checkout -q FETCH_HEAD')
os.chdir(REPO_DIR)
sh('git lfs install --local --skip-smudge')
COMMIT = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
print('commit', COMMIT)
# 로컬과 같은 transformers 버전으로 맞춥니다. torch는 Colab에 설치된 CUDA 버전을 그대로 씁니다.
# baselines extra(CredSweeper)는 학습 끝의 Rule 비교에 필요합니다.
sh("pip install -q transformers==4.57.6 && pip install -q -e '.[baselines]'")
# 학습 데이터가 git에 없으면 여기서 다시 만듭니다(PREPARE).
for command in PREPARE:
    sh(command)

In [ ]:
# 시작 가중치 준비
REVISION = 'abc32620dd4f6ab06f5fbe905dc25f310618e09f'  # 모든 실험이 쓰는 mmBERT-small 고정 revision
if START_FROM in ('v9', 'v10'):
    MODEL_NAME = f'models/span-detector-{START_FROM}'
    sh(f'git lfs pull --include={MODEL_NAME}/model.safetensors')
    if os.path.getsize(f'{MODEL_NAME}/model.safetensors') < 100_000_000:
        raise RuntimeError('LFS 가중치를 받지 못했습니다(포인터 파일만 있음).')
elif START_FROM == 'backbone':
    from huggingface_hub import snapshot_download
    MODEL_NAME = 'runs/backbones/mmbert-small'
    snapshot_download('jhu-clsp/mmBERT-small', revision=REVISION, local_dir=MODEL_NAME,
                      allow_patterns=['*.json', 'pytorch_model.bin'])
else:
    raise ValueError("START_FROM은 'v9', 'v10', 'backbone' 중 하나여야 합니다.")
print('시작 가중치:', MODEL_NAME)

In [ ]:
# 학습 설정 만들기: BASE_CONFIG에 장치, 시작 가중치, 저장 위치, OVERRIDES, seed만 바꿉니다.
import json
import yaml
with open(BASE_CONFIG) as f:
    config = yaml.safe_load(f)
config.update(model_name=MODEL_NAME, device='cuda', local_files_only=True)
config.update(OVERRIDES)
config['colab'] = dict(gpu=GPU_NAME, start_from=START_FROM, base_config=BASE_CONFIG, repo_ref=REPO_REF,
                       commit=COMMIT, overrides=OVERRIDES, torch=str(torch.__version__))
os.makedirs('runs/colab', exist_ok=True)
CONFIG_PATHS = {}
for name, seed in RUNS:
    run_config = dict(config, output_dir=f'{DRIVE_DIR}/{name}')
    if seed is not None:
        run_config['seed'] = seed
    CONFIG_PATHS[name] = f'runs/colab/{name}.yaml'
    with open(CONFIG_PATHS[name], 'w') as f:
        yaml.safe_dump(run_config, f, sort_keys=False, allow_unicode=True)
print(open(CONFIG_PATHS[RUNS[0][0]]).read())

In [ ]:
# 학습. seed가 여러 개면 차례로 학습합니다. 진행 로그는 Drive의 <이름>.log에도 저장됩니다.
# USE_TF=0: Colab에 함께 설치된 TensorFlow를 불러오지 않습니다(혼동되는 'GPU will not be used' 메시지 방지).
for name, path in CONFIG_PATHS.items():
    sh(f'USE_TF=0 PYTHONPATH=src python scripts/span_pipeline.py train --config {path} 2>&1 | tee {DRIVE_DIR}/{name}.log')

In [ ]:
# validation 결과만 봅니다(test 결과는 열지 않음).
for name, _ in RUNS:
    print('==', name)
    with open(f'{DRIVE_DIR}/{name}/history.json') as f:
        history = json.load(f)
    for h in history:
        v = h['validation']
        print(f"epoch {h['epoch']}  loss {h['loss']:.2e}  cutoff {h['threshold']}  "
              f"TP/FP/FN {v['tp']}/{v['fp']}/{v['fn']}  precision {v['precision'] or 0:.3f}  F2 {v['f2'] or 0:.3f}")
    with open(f'{DRIVE_DIR}/{name}/model/span_bundle.json') as f:
        print('선택된 epoch:', json.load(f)['best_epoch'])

In [ ]:
# 내려받을 zip 만들기: 모델과 validation 기록만 넣습니다(test 평가 파일 제외).
import shutil
import tempfile
KEEP = ['model', 'history.json', 'config.json', 'alignment.json']
for name, _ in RUNS:
    with tempfile.TemporaryDirectory() as staging:
        for item in KEEP:
            source = f'{DRIVE_DIR}/{name}/{item}'
            if os.path.isdir(source):
                shutil.copytree(source, f'{staging}/{item}')
            else:
                shutil.copy(source, staging)
        archive = shutil.make_archive(f'{DRIVE_DIR}/{name}', 'zip', staging)
    print('zip:', archive, round(os.path.getsize(archive) / 2**20), 'MB')
DATA_DIR = config['data_dir']
print('\n로컬 저장소에서 이어서 실행할 명령(자세한 설명은 docs/ml_span/COLAB_TRAINING.md):')
for name, _ in RUNS:
    print(f"""
mkdir -p runs/{name} && unzip ~/Downloads/{name}.zip -d runs/{name}
PYTHONPATH=src .venv/bin/python scripts/validate_span_boundaries.py --checkpoint runs/{name}/model --data-dir {DATA_DIR} --output runs/{name}-boundary/model --report reports/ml_span/{name}-boundary-validation.json --boundary-dir runs/span-boundary-v3""")

## 다음 단계(로컬)

zip을 Drive에서 내려받은 뒤 로컬 저장소에서 `docs/ml_span/COLAB_TRAINING.md`의 순서대로 진행합니다.

1. 경계 보정 검증(위 셀이 출력한 명령)
2. bundle에 런타임 필터 `contextual-function-words-v1` 추가
3. 현재 기본 모델 `models/span-detector-v10`을 기준으로 validation에서 cutoff 선택
4. **새 독립 hold-out을 먼저 만든 뒤** 비교와 gate 검사. v10 test와 hold-out v11은 이미 결과를 본 개발셋입니다.